In [55]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
pd.set_option('display.max_columns', None)
df = pd.read_csv("../data/processed/Accident_auto_combined.csv")
#secu



C:\Users\nvann\AppData\Local\Temp\ipykernel_57136\4037019965.py:6: DtypeWarning: Columns (1,2,14,17,21,24,25,27,28,44,46,47,53,54,55,56,59) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/Accident_auto_combined.csv")


In [56]:

import re


def resume_dataframe(df):
    summary = []

    for col in df.columns:
        col_data = df[col]
        total = len(col_data)
        dtype = col_data.dtype
        na_pct = 100 * col_data.isna().sum() / total
        isnull_pct = 100 * col_data.isnull().sum() / total

        # Pourcentage de valeurs à -1
        minus1_count = (col_data == -1).sum()
        minus1_pct = 100 * minus1_count / total

        # Mode et son % d'apparition
        if not col_data.dropna().empty:
            mode = col_data.mode().iloc[0]
            mode_pct = 100 * (col_data == mode).sum() / total
        else:
            mode = None
            mode_pct = None

        summary.append({
            'nom_col': col,
            'Type': dtype,
            '% NA': round(na_pct, 2),
            '% isnull': round(isnull_pct, 2),
            '% -1': round(minus1_pct, 2),
            'nb -1': int(minus1_count),
            'Valeur dominante': mode,
            '% dominante': round(mode_pct, 2) if mode_pct is not None else None
        })
    return pd.DataFrame(summary)

In [58]:
def min_max_year_per_column(df, year_col='an', missing_codes=None):
    if missing_codes is None:
        missing_codes = [-1, 9]  # codes manquants à ignorer

    results = []
    
    # On s'assure que la colonne année est bien présente
    if year_col not in df.columns:
        raise ValueError(f"La colonne année '{year_col}' n'existe pas dans le DataFrame.")

    # Remplacement des codes manquants dans la colonne année
    years = df[year_col].replace(missing_codes, np.nan)
    years = years.dropna()

    for col in df.columns:
        #if col == year_col:
        #    continue  # on ne regarde pas la colonne année elle-même

        # Colonne nettoyée des codes manquants
        col_clean = df[col].replace(missing_codes, np.nan)

        # Indices où la colonne a une donnée valide (non-NA)
        valid_idx = col_clean.notna()

        # Années correspondantes où on a une donnée valide
        valid_years = years[valid_idx]

        if valid_years.empty:
            year_min = None
            year_max = None
        else:
            year_min = int(valid_years.min())
            year_max = int(valid_years.max())

        results.append({
            'nom_col': col,
            'année_min_data': year_min,
            'année_max_data': year_max
        })

    return pd.DataFrame(results)

In [60]:
#Création du résumé des données
# et de la fonction de fusion des DataFrames
# Création d'une fonction pour fusionner les DataFramesall
# je veux merger resume_dataframe(df) et  min_max_year_per_column(df) sur la colonne variable
def merge_dataframes(df1, df2,df1_on_variable,df2_on_variable):
        return pd.merge(df1, df2, left_on=df1_on_variable, right_on=df2_on_variable, how='outer') 

merged_df = merge_dataframes(resume_dataframe(df), min_max_year_per_column(df), 'nom_col', 'nom_col')




In [54]:
merged_df.head(75)

,nom_col,Type,% NA,% isnull,% -1,nb -1,Valeur dominante,% dominante,année_min_data,année_max_data
0,Num_Acc,int64,0.00,0.00,0.00,0,202300008814,0.01,2005,2023
1,actp,object,1.70,1.70,0.00,0,0.0,53.19,2005,2023
2,adr,object,10.02,10.02,0.00,0,AUTOROUTE A1,0.44,2005,2023
3,agg,int64,0.00,0.00,0.00,0,2,64.52,2005,2023
4,an,int64,0.00,0.00,0.00,0,2023,9.94,2005,2023
5,an_nais,float64,0.65,0.65,0.00,0,1988.0,2.31,2005,2023
6,atm,float64,0.00,0.00,0.00,82,1.0,80.29,2005,2023
7,catr,float64,0.00,0.00,0.00,0,4.0,44.89,2005,2023
8,catu,int64,0.00,0.00,0.00,0,1,74.43,2005,2023
9,catv,int64,0.00,0.00,0.00,36,7,64.35,2005,2023
